In [ ]:
# If running within Google Colab it sets up the environment
# Please enable GPU under Runtime > Change runtime type

import os, sys

if 'google.colab' in sys.modules:
    repo_url = "https://github.com/mauro-m-monsalve/NeuralGeometry.git"
    repo_dir = "/content/NeuralGeometry"
    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    os.chdir(repo_dir)
    
    # # Optionally install dependencies not included in colab
    # !pip install pot

#
# Neural Retinotopic Model of Decision-Making
#


This notebook analyzes a mechanistic model of perceptual decision-making based on **competitive dynamics of activity bumps** in a retinotopically organized 2D neural field. 

The model accurately reproduces the full **geometry of the decision manifold** observed in neural population data, including:

- The **low intrinsic dimensionality** of neural trajectories,
- The **organization of trials by reaction time** across the manifold,
- The distinct **deliberation-to-commitment transition**,
- The **curvature** and **separability** of trajectories by choice,
- The **tortuosity** of single-trial paths,
- The decomposition of local fluctuations into behaviorally relevant directions — **resolution** (decision progress toward action selection) and **uncertainty** (affecting decision time and choice selection).

Beyond reproducing these manifold properties, the model predicts a **retinotopic shift** in the bump of neural activity depending on **decision progress** and **evidence strength** (quantified by reaction time). 

These predicted shifts lead to two experimental predictions that we verify in neural and behavioral data:
- The **retinotopic organization of single-neuron choice selectivity**,
- The **dependence of saccadic endpoint distributions** on reaction time.

Finally, by analyzing the model's local dynamics, we uncover how **momentary evidence is integrated** across the manifold and how this integration depends on both **retinotopy** and the **stage of the decision**.

Overall, the model provides a **mechanistic link** between population geometry, sensory evidence integration, and retinotopic motor behavior during decision-making.

In [ ]:
import os
import sys

# Set the root directory

# Automatically find the project root (directory containing 'src' or 'data')
def find_project_root(marker_dirs=("src", "data","notebooks")):
    path = os.getcwd()
    while path != "/" and not all(os.path.exists(os.path.join(path, d)) for d in marker_dirs):
        path = os.path.dirname(path)
    return path

PROJECT_ROOT = find_project_root()

# Set up Python import path and working directory
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

#
# DecisionMakingModel: Simulation Overview

> **Summary:**  
> The `DecisionMakingModel` simulates population neural dynamics during perceptual decision-making, organized in a **retinotopic 2D neural field**. Neurons interact through structured excitatory and inhibitory recurrent connectivity, and receive spatially localized sensory input and noise. Decisions emerge as bumps of activity crossing defined target regions.

---

## 1. Neural Field Structure

The model consists of two populations:
- **Excitatory (E)** neurons
- **Inhibitory (I)** neurons

arranged on a 2D spatial grid of size $N_x \times N_y$.

Each neuron at position $(x,y)$ has an excitatory rate $r_e(x,y,t)$ and an inhibitory rate $r_i(x,y,t)$.

---

## 2. Dynamic Equations

The neural dynamics are governed by discretized differential equations:

$$
\tau_e \frac{dr_e}{dt} = -r_e + k_e \left[ \left( W_{ee} \ast r_e \right) - \left( W_{ei} \ast r_i \right) + I_{\text{ext}} + \eta \right]_+^{n_e}
$$

$$
\tau_i \frac{dr_i}{dt} = -r_i + k_i \left[ \left( W_{ie} \ast r_e \right) - \left( W_{ii} \ast r_i \right) + I_{\text{ext}} \right]_+^{n_i}
$$

where:
- $\tau_e, \tau_i$ are time constants,
- $W_{ab}$ are structured recurrent connectivity matrices between populations $b \to a$,
- $I_{\text{ext}}$ is the external deterministic (mean) input (task condition-specific),
- $\eta$ is spatially and temporally correlated noise,
- $[\cdot]_+$ denotes rectification (ReLU),
- $k_e, k_i$ are gain factors,
- $n_e, n_i$ are nonlinear exponents on rectified inputs.

Time is discretized with step size $\Delta t$, and the system is updated iteratively over $n_{\text{steps}}$.

---

## 3. Connectivity Kernels

Each recurrent connectivity matrix $W_{ab}$ is **Gaussian-shaped**:

$$
W_{ab}(d) = w_{ab} \, \exp\left( -\frac{d^2}{s_{ab}^2} \right)
$$

where:
- $d$ is the Euclidean distance (with optional wrap-around boundaries),
- $w_{ab}$ is the maximal strength,
- $s_{ab}$ is the spread (standard deviation) of connectivity.

All kernels are normalized row-wise to ensure constant total input.

---

## 4. External Inputs

The external sensory input is spatially localized around **target locations**.

For each target $i$ with center at $(x_i, y_i)$:

$$
I_{\text{target},i}(x,y) = \exp\left( -\frac{(x-x_i)^2 + (y-y_i)^2}{\sigma_{\text{input}}^2} \right)
$$

For each trial the conditions prefer a certain target. The total external input for that trial is:

$$
I_{\text{ext}}(x,y) = c \times \left( (1 + g m) \cdot I_{\text{target, preferred}}(x,y) + \text{(balanced gain on other targets)} \right)
$$

where:
- $c$ is a global input scaling factor,
- $g$ is a gain factor,
- $m$ is the dot-motion coherence level,
- When `balanced_gain=True`, non-preferred targets optionally have a reduced gain $-g/(N-1)$, with $N$ the total number of targets. Otherwise when `balanced_gain=False`, non-preferred targets have $g=0$.

---

## 5. Noise Structure

Noise $\eta(x,y,t)$ is:
- **Temporally correlated** via an Ornstein–Uhlenbeck process with time constant $\tau_{\text{noise}}$,
- **Spatially correlated** using a Gaussian kernel across the retinotopic sheet.

Specifically:

$$
d\eta = -\frac{1}{\tau_{\text{noise}}}\eta\,dt + \sqrt{\frac{2}{\tau_{\text{noise}}}} \sigma_{\text{noise}} dW
$$

where $dW$ is a Wiener process (white noise).

Spatial correlation is enforced by multiplying the noise by a spatial smoothing matrix.

---

## 6. Decision Termination

A **decision** is made when the neural activity bump sufficiently differentiates between targets.

- Compute mean activity inside **target-in** (Tin) masks over time.
- Decision is terminated when the maximal difference between any two Tin activities exceeds a threshold:

$$
\Delta r_{\text{Tin}}(t) = \max_i r_{\text{Tin},i}(t) - \min_j r_{\text{Tin},j}(t)
$$

Decision time is the first $t$ satisfying:

$$
\Delta r_{\text{Tin}}(t) > \text{threshold}
$$


---

## 7. Retinotopy and Saving

- Spatial snapshots of neural activity are saved (cropped if needed).
- Decision-related dynamics can be projected into lower-dimensional spaces (e.g., PCA) for analysis.
- Various utilities allow plotting:
  - Trial trajectories,
  - Tin curves,
  - Full retinotopic heatmaps,
  - Choice selectivity maps.

---

## 8. Utility Features

- **Flexible device placement**: CPU, CUDA (GPU), or MPS (Mac GPUs).
- **Parallel execution**: Full parallelization across simulation conditions.
- **Configuration loading**: All parameters specified via a YAML configuration.
- **Data output**: Final outputs saved as pandas DataFrames for downstream analysis.

---

## Key Insights from Model Behavior

- The model shows **population-level bump dynamics** naturally arising from connectivity.
- Simulates **choice formation** as a distributed competition between target locations.
- Predicts:
  - **Reaction time distributions**,
  - **Error rates**,
  - **Choice selectivity retinotopy**,
  - **Population trajectory organization**.
- Provides a mechanistic explanation for **spatially organized evidence integration** during decision making.

---
#
#

In [ ]:
import yaml

# Load the config
with open("data/model_config.yaml", 'r') as f:
    config = yaml.safe_load(f)

# # Example modifications
# config['simulation']['NTrials'] = 200                # set number of trials per condition


# # Save a modified yaml file for experimenting
# with open("data/model_config_run1.yaml", 'w') as f:
#     yaml.dump(config, f, sort_keys=False)

# Print the whole config file
print(yaml.dump(config, sort_keys=False, default_flow_style=False))


In [ ]:

from src.model import DecisionMakingModel

model = DecisionMakingModel(params='data/model_config.yaml')

model.initialize()

# # Use parallel run to simulate all conditions in parallel with joblib. Faster depending on available hardware.
# model.parallel_run()

model.run()

# Plots a random trial from the condition set (before decision termination)
model.plot_trial(motion_coherence=0.0)

# Terminates the decision
df = model.terminate_decision(remove_outliers=1.)


In [ ]:

from src.plot_utils import plot_violin, plot_proportion

plot_violin(df, var='RT', group_by='motion_coherence', color_by='choice').show()
plot_proportion(df, choice='Contra', targets='target',group_by='motion_coherence')


In [ ]:
from scipy.ndimage import gaussian_filter1d
import numpy as np

# Parameters
Cut = 30     # time steps (10ms bins) to trim at the start
CutEnd = -1  # time steps before saccadeDetected to trim at the end
sigma = 5    # smoothing kernel width

# Smooth 're' trajectories first
for trial in df.index:
    df.at[trial, 're'] = gaussian_filter1d(df.at[trial, 're'], sigma=sigma, axis=1)

# Compute reference starting points for alignment
# Aligment is left here for comparison to the analysis of the LIP recordings, but not neccessary for the model.
start_vectors = np.stack([df.at[trial, 're'][:, 0] for trial in df.index], axis=1)
mean_start = np.mean(start_vectors, axis=1, keepdims=True)

# Align and cut 're', 'input', and optionally 'ri'
for trial in df.index:
    # Align 're'
    traj_re = df.at[trial, 're']
    aligned_re = traj_re[:, Cut:CutEnd] - traj_re[:, [0]] + mean_start
    df.at[trial, 're'] = aligned_re

    # Cut 'input'
    traj_input = df.at[trial, 'input']
    aligned_input = traj_input[:, Cut:CutEnd] # No alignment
    df.at[trial, 'input'] = aligned_input

    # Cut 'ri' if it exists
    if 'ri' in df.columns and isinstance(df.at[trial, 'ri'], np.ndarray):
        traj_ri = df.at[trial, 'ri']
        aligned_ri = traj_ri[:, Cut:CutEnd]
        df.at[trial, 'ri'] = aligned_ri

In [ ]:

from src.geometry import get_pca

pca,scaler = get_pca(df, column='re', cells=None, components=10, plot=True)

In [ ]:

from src.geometry import compute_geometry_measures

compute_geometry_measures(df, column='re-PCA', timeres=10, arc_res=101, w_curvature=2)
df.keys()[-12:]



In [ ]:
from src.geometry import local_average

speed_0 , nba_0 = local_average(df[df['choice']=='Contra'], column='Speed-Arc', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=None)
tortuosity_0 , nba_0 = local_average(df[df['choice']=='Contra'], column='Tortuosity-Arc', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=None)


In [ ]:

from src.plot_utils import plot_local_average

plot_local_average(data=speed_0, behavioral_axis=nba_0, var_name='Speed (Hz/ms)').show()
plot_local_average(data=tortuosity_0, behavioral_axis=nba_0, var_name='Tortuosity').show()


In [ ]:
from src.geometry import local_average

# choice 0 and 1 correspond to contralateral and ipsilateral choice respectively
pop_0 , nba_0 = local_average(df[df['choice']=='Contra'], column='re-PCA-Arc', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=None)
pop_1 , nba_1 = local_average(df[df['choice']=='Ipsi'], column='re-PCA-Arc', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=None)


In [ ]:
from src.plot_utils import plot_local_average_pop

fig = plot_local_average_pop(
    manifolds=[pop_0,pop_1],
    behavioral_axes=[nba_0,nba_1],
    color_ranges=[("gray blue", "neon blue"), ("gray pink", "neon pink")],
    names=['Contralateral Choice', 'Ipsilateral Choice']
)
fig.show()
#fig.write_html('AverageManifold.html')

In [ ]:
from src.geometry import K

distance_0 = np.linalg.norm(pop_0-pop_1,axis=0)
curvature_0 = 180/np.pi*np.stack([K(pop_0[:, :, i],w=1) for i in range(pop_0.shape[2])], axis=1)


from src.plot_utils import plot_local_average

plot_local_average(data=distance_0, behavioral_axis=nba_0/np.max(nba_0), var_name='Distance (Hz)',title='Local Distance between Choice-Submanifolds',ba_title='Reaction Time (normalized)').show()
plot_local_average(data=curvature_0, behavioral_axis=nba_0, var_name='Curvature (deg/Hz)',title='Submanifold Curvature in the Resolution Direction').show()


In [ ]:
from src.geometry import tangent_space
from scipy.ndimage import gaussian_filter as gf2

# compute unit tangent vectors
t0 = tangent_space(pop_0, axis=1)
t1 = tangent_space(pop_1, axis=1)

dot = np.einsum('ntm,ntm->tm', t0, t1)
norms = np.linalg.norm(t0, axis=0) * np.linalg.norm(t1, axis=0)
norms[norms == 0] = np.nan 

# compute cosine similarity between unit tangent vectors
cosine_similarity = gf2(dot / norms, sigma=5)




from src.plot_utils import plot_local_average

plot_local_average(data=cosine_similarity, behavioral_axis=nba_0/np.max(nba_0), heatmap_scale='icefire', var_name='Cosine Similarity',title='Local Cosine Similarity between Choice-Submanifolds',ba_title='Reaction Time (normalized)').show()


In [ ]:
from src.geometry import tangent_space
from src.geometry import compute_unit_orthogonal_vectors
from src.geometry import get_components

# compute reference tangent spaces
tangent_0 = tangent_space(pop_0, axis=1) # along arc-length
transverse_0 = -tangent_space(pop_0, axis=2) # along reaction-time
transverse_0 = compute_unit_orthogonal_vectors(tangent_0, transverse_0, axis=0)

tangent_1 = tangent_space(pop_1, axis=1) # along arc-length
transverse_1 = -tangent_space(pop_1, axis=2) # along reaction-time
transverse_1 

# Compute squared components of projections of the single-trial unit tangent vectors into the resolution and uncertainty directions of the average manifold
get_components(df, tangent_ref=tangent_0, conditions=[df['choice']=='Contra'], new_behavioral_axis=nba_0, name='Resolution',trajectories='re-PCA-Arc')
get_components(df, tangent_ref=transverse_0, conditions=[df['choice']=='Contra'], new_behavioral_axis=nba_0, name='Uncertainty',trajectories='re-PCA-Arc')
get_components(df, tangent_ref=tangent_1, conditions=[df['choice']=='Ipsi'], new_behavioral_axis=nba_1, name='Resolution',trajectories='re-PCA-Arc')
get_components(df, tangent_ref=transverse_1, conditions=[df['choice']=='Ipsi'], new_behavioral_axis=nba_1, name='Uncertainty',trajectories='re-PCA-Arc')


df['OffManifoldSquareProjection'] = 1 - df['ResolutionSquareProjection'] - df['UncertaintySquareProjection']


resolution_0 , nba_0 = local_average(df[df['choice']=='Contra'], column='ResolutionSquareProjection', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=None)
uncertainty_0 , nba_0 = local_average(df[df['choice']=='Contra'], column='UncertaintySquareProjection', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=None)
offmanifold_0 , nba_0 = local_average(df[df['choice']=='Contra'], column='OffManifoldSquareProjection', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=None)

In [ ]:
from src.plot_utils import plot_local_average

plot_local_average(data=resolution_0, behavioral_axis=nba_0, var_name='Component',title='Squared Projection in the Resolution Direction').show()
plot_local_average(data=uncertainty_0, behavioral_axis=nba_0, var_name='Component',title='Squared Projection in the Uncertainty Direction').show()
plot_local_average(data=offmanifold_0, behavioral_axis=nba_0, var_name='Component',title='Squared Projection Off-Manifold').show()


To reconstruct the full retinotopy of the decision manifold, we first computed local averages in a lower-dimensional PCA space, now we invert the transformation back to the full pixel space. This approach avoids the computational cost and instability of averaging directly in the original high-dimensional input space, where most dimensions are empty or noisy.

In [ ]:

from src.geometry import inverse_pca
import pandas as pd
pop_0_full = inverse_pca(pca, pop_0, scaler)
pop_1_full = inverse_pca(pca, pop_1, scaler)

pops = pd.DataFrame([{'session':'S', 'choice':'Contra', 'pop_locav':pop_0_full},{'session':'S', 'choice':'Ipsi', 'pop_locav':pop_1_full}])

In [ ]:
vf_plot_params={'shape':(model.crop_Nx,model.crop_Ny), # grid shape to reshape first dimension
             'extent':model.cfg['simulation']['extent_save'], # [X,Y] form -X/2 to X/2 and -Y/2 to Y/2
             'n_points': (6,6), # equally spaced number of (arc,rt) points in the manifold coordinates to plot
             'new_behavioral_axis':nba_0, # new behavioral axis (interpolated RT) to plot
             'nba_name': 'Reaction Time (ms)', # name of the new behavioral axis
             'cmap':'inferno', # colormap, use 'icefire' for a divergent colormap with data centered at zero
             'title': 'Retinotopy of the Local Average Submanifold of Contralateral Choice', # title of the plot
             'targets': [{'center':model.cfg['input']['targets'][0]['center'], 'name':model.cfg['input']['targets'][0]['name'], 'color': 'neon blue'},
                         {'center':model.cfg['input']['targets'][1]['center'], 'name':model.cfg['input']['targets'][1]['name'], 'color': 'neon pink'}],
             'tin_radius': model.cfg['simulation']['tin_radius'], # radius of the target in pixels
             }


from src.plot_utils import plot_manifold_retinotopy

plot_manifold_retinotopy(pop_0_full, **vf_plot_params).show()

## Choice Selectivity in the Model

> **Summary:**  
> We simulate the decision-making model to study the retinotopic organization of **choice selectivity** — how different regions of the neural population become selective for different choices during decision formation.

---

### Method Overview

- For each model simulation, we collect neural population activity across trials leading to different choices.
- **Choice selectivity** is computed **not as a simple difference of mean activities**, but as a **decorrelated similarity measure**:
  - Specifically, at each point along the decision manifold (arc length), a running window is taken across arc.
  - Within this window, **the Pearson correlation** between the population activity patterns for the two choices is computed.
  - The **choice selectivity** is then defined as:

    $$
    \text{Choice Selectivity} = \frac{1 - \text{Correlation}}{2}
    $$

  - This scales from 0 (identical patterns) to 1 (perfectly anti-correlated patterns), ensuring a normalized and sensitive measure of choice-specific structure.
- **Arc windowing:**  
  The correlation is computed within **local windows along arc length** (default ~20%), ensuring that the measure captures local differences rather than global trends.

---

### Practical Implementation Details

- **Multiple model runs:**  
  In the main paper (Figure C), **100 independent simulations** were run with the same model parameters to build robust statistics.

- **Histograms of selectivity differences:**  
  The differences in choice selectivity between conditions (e.g., fast vs slow reaction times) are **collated across all runs** to build smooth density estimates.

- **Retinotopic maps:**  
  Retinotopic maps of choice selectivity are **averaged across simulations** to reveal consistent spatial patterns.

- **Current notebook:**  
  Here, we illustrate the choice selectivity from **one instance** of the model only for clarity. Therefore, variability and fine-scale structure may differ slightly from the averaged results shown in the paper.



In [ ]:
from src.properties import choice_selectivity_binary

cs = choice_selectivity_binary(pop_0_full, pop_1_full, arc_window=0.2)

In [ ]:

from src.plot_utils import plot_local_average
from src.model import get_pixel_at

loc = np.array(model.cfg['input']['targets'][0]['center']) + np.array([model.cfg['simulation']['tin_radius'],0])
example_cell_index = get_pixel_at(loc=loc, extent=model.cfg['simulation']['extent_save'], grid_shape=(model.crop_Nx, model.crop_Ny) )

plot_local_average(pop_0_full[example_cell_index], var_name='Firing Rate (Hz)', title = 'Activity of the example cell for the contra-lateral choice').show()
plot_local_average(pop_1_full[example_cell_index], var_name='Firing Rate (Hz)', title = 'Activity of the example cell for the ipsi-lateral choice').show()
plot_local_average(cs[example_cell_index],heatmap_scale='earth_r',smooth_sigma=None,var_name='Choice Selectivity', title = 'Choice Selectivity in running Arc-Length Window').show()

In [ ]:
from src.properties import choice_selectivity_binary
import pandas as pd
import numpy as np


cs_data = []

sessions = pops['session'].unique()
choices = pops['choice'].unique()

for session in sessions:
    pop0 = pops[(pops['session'] == session) & (pops['choice'] == choices[0])]['pop_locav'].values[0]
    pop1 = pops[(pops['session'] == session) & (pops['choice'] == choices[1])]['pop_locav'].values[0]

    # Compute choice selectivity
    cs = choice_selectivity_binary(pop0, pop1, arc_window=0.2)

    # Compute per-choice ranges
    r0 = np.max(pop0, axis=(1, 2)) - np.min(pop0, axis=(1, 2))
    r1 = np.max(pop1, axis=(1, 2)) - np.min(pop1, axis=(1, 2))

    # The minimum range across both choices
    #cell_range = np.minimum(r0, r1)
    cell_range = np.maximum(r0, r1)

    cs_data.append({
        'session': session,
        'choice_selectivity': cs,
        'cell_range': cell_range
    })

# Convert to DataFrame
cs_df = pd.DataFrame(cs_data)

In [ ]:
from src.plot_utils import plot_pop_selectivity_arcslice

plot_pop_selectivity_arcslice(cs_df, session='S', arc=0.9, cell_range_threshold=10.)

In [ ]:

from src.plot_utils import plot_pop_selectivity_scatter

# open_interactive=True will send you to another browser tab for full HTML interactivity
plot_pop_selectivity_scatter(cs_df, pops, arc=0.9, percentile=1, cell_range_threshold=1.0, open_interactive=False)

In [ ]:

from src.properties import get_pop_cs

cs_slow,cs_fast = get_pop_cs(cs_df, session='S', arc=0.9, percentile=1)


vf_plot_params={'shape':(model.crop_Nx,model.crop_Ny), # grid shape to reshape first dimension
             'extent':model.cfg['simulation']['extent_save'], # [X,Y] form -X/2 to X/2 and -Y/2 to Y/2
             #'cmap':'earth_r' centers at .5 , use 'icefire' for a divergent colormap with data centered at zero
             #'contour': True,
             'smooth_sigma': 1, # smoothing kernel width in dva
             'zoom': 10,
             'targets': [{'center':model.cfg['input']['targets'][0]['center'], 'name':model.cfg['input']['targets'][0]['name'], 'color': 'neon blue'},
                         {'center':model.cfg['input']['targets'][1]['center'], 'name':model.cfg['input']['targets'][1]['name'], 'color': 'neon pink'}],
             'tin_radius': model.cfg['simulation']['tin_radius'], # radius of the target in pixels
             }

from src.plot_utils import plot_property_retinotopy

plot_property_retinotopy(cs_slow, **vf_plot_params, contour=True, title='Retinotopy of Choice Selectivity Slow').show()
plot_property_retinotopy(cs_fast, **vf_plot_params, contour=True, title='Retinotopy of Choice Selectivity Fast').show()
plot_property_retinotopy(cs_fast - cs_slow, **vf_plot_params,cmap='icefire', title='Retinotopy of Choice Selectivity Difference').show()



## Local Resolution- and Uncertainty-driving Input Directions

> **Summary:**  
> We fit local linear models across the decision manifold to reveal how spatial fluctuations in the input drive fluctuations in neural dynamics along behaviorally meaningful directions (resolution and uncertainty).

In this section, we investigate how local fluctuations in the input to the neural population relate to fluctuations in the neural activity trajectories — specifically, in the directions associated with decision *Resolution* and *Uncertainty*.

To do this, we perform **local linear fits** across the manifold, using patches defined by small 2D windows in normalized arc-length (progress along the decision manifold) and normalized reaction time.

---

### Method Overview

- For each local patch (Arc × RT window), we extract:
  - The fluctuations in the **input activity** (after removing its local mean in the patch and normalizing its magnitude).
  - The **resolution** and **uncertainty** projections of the neural activity trajectory (the tangent vector components along behaviorally meaningful directions).

- We fit a **separate linear model** for each window:
  - The model predicts resolution and uncertainty projections from the input fluctuations.
  - Two sets of input weights are learned per window: one driving resolution and one driving uncertainty.

- **Smoothing** is enforced by adding a penalty on the second spatial derivative of the weights across the retinotopic sheet (favoring spatially smooth weight maps).

- **Early stopping** prevents overfitting by monitoring convergence during training.

---

### Why this approach?

- Neural fluctuations around the decision manifold are expected to be **low-dimensional** and behaviorally structured.
- By fitting local linear models, we test the idea that **input fluctuations in specific spatial patterns** systematically drive variability in:
  - The **rate of decision evolution** (*Resolution*).
  - The **sensitivity of the neural state to perturbations near the decision boundary** (*Uncertainty*), which influences the variability and speed of reaction times (higher absolute uncertainty typically corresponds to slower and more variable decisions).

- In essence, **the learned input weight maps reveal the retinotopic patterns that control local behavioral dynamics** during decision formation.

---

### What does this analysis reveal?

- **Structured input control:**  
  If the fits are accurate and spatially organized, it suggests that **localized input fluctuations** (e.g., noisy sensory evidence) are directly responsible for modulating decision dynamics within the neural population.

- **Input-driven evidence accumulation:**  
  By projecting the fitted weights onto a known *evidence direction* in the retinotopic sheet, we can quantify how much of the neural variability is affected by external sensory evidence.

- **Deliberation-to-commitment transition:**  
  Critically, this analysis reveals that:
  - **Early in the decision process** (lower arc-length), **momentary evidence fluctuations strongly drive neural fluctuations** along the uncertainty axis, supporting active deliberation.
  - **Later in the decision process** (higher arc-length), **the effect of input fluctuations weakens**, indicating a transition toward **commitment**, where the neural trajectory becomes more resistant to noise.

This provides direct evidence for a **dynamic shift from deliberation to commitment** during decision-making in the model.

In [ ]:
# Get the global evidence direction favoring the contralateral target

target_contra, target_ipsi = model.target_inputs.cpu().numpy()
evdir = target_contra - target_ipsi
evdir = evdir/np.linalg.norm(evdir)

plot_property_retinotopy(evdir, **vf_plot_params,cmap='icefire', title='Retinotopy of the Contra Evidence Direction').show()


In [ ]:
# Compute resolution and uncertainty tangent spaces
tangent_0 = tangent_space(pop_0_full, axis=1) # along arc-length
transverse_0 = -tangent_space(pop_0_full, axis=2) # along reaction-time
transverse_0 = compute_unit_orthogonal_vectors(tangent_0, transverse_0, axis=0)

cosine_similarity_res = np.tensordot(evdir, tangent_0, axes=([0], [0]))
cosine_similarity_unc = np.tensordot(evdir, transverse_0, axes=([0], [0]))

from src.plot_utils import plot_local_average
plot_local_average(data=cosine_similarity_res, behavioral_axis=nba_0, heatmap_scale='icefire', var_name='Cosine Similarity',title='Contra Evidence Direction vs. Resolution tanget Space').show()
plot_local_average(data=cosine_similarity_unc, behavioral_axis=nba_0, heatmap_scale='icefire', var_name='Cosine Similarity',title='Contra Evidence Direction vs. Uncertainty tanget Space').show()



In [ ]:

from src.geometry import reparametrize_to_arc

# Adds the column 'input-Arc' to the dataframe in place
reparametrize_to_arc(df,column='input')

In [ ]:

# # Uncomment to run one instance of the local fit
# from src.model import fit_local_linear_model
# df_fit = fit_local_linear_model(df, device='cpu')

# We ran the previous fit 100 times with different random seeds and averaged the results over runs. 
# This averages out noisy fits in empty pixel regions where there was no information to fit.

import gzip, pickle
with gzip.open('data/local_evidence_fit.pkl.gz', 'rb') as f:
    df_fit = pd.DataFrame(pickle.load(f))

print(df_fit.keys())
print(f"Mean bias for the resolution fit: {df_fit['bias_resolution'].mean():.2f}")
print(f"Mean bias for the uncertainty fit: {df_fit['bias_uncertainty'].mean():.2f}")


In [ ]:

vf_plot_params={'shape': (model.crop_Nx,model.crop_Ny), # grid shape to reshape first dimension
             'extent': model.cfg['simulation']['extent_save'], # [X,Y] form -X/2 to X/2 and -Y/2 to Y/2
             #'cmap':'earth_r' centers at .5 , use 'icefire' for a divergent colormap with data centered at zero
             #'contour': True,
             'smooth_sigma': None, # smoothing kernel width in dva
             'zoom': None,
             'targets': [{'center':model.cfg['input']['targets'][0]['center'], 'name':model.cfg['input']['targets'][0]['name'], 'color': 'neon blue'},
                         {'center':model.cfg['input']['targets'][1]['center'], 'name':model.cfg['input']['targets'][1]['name'], 'color': 'neon pink'}],
             'tin_radius': model.cfg['simulation']['tin_radius'], # radius of the target in pixels
             }

from src.plot_utils import plot_local_fit
fig = plot_local_fit(df_fit, column='weights_uncertainty', **vf_plot_params)
fig.show()

In [ ]:

from src.model import add_ev_modulation
from src.plot_utils import plot_evidence_modulation

df_fit = add_ev_modulation(df_fit,evdir)
plot_evidence_modulation(df_fit)
